In [1]:
from pathlib import Path
import time, pandas as pd
from espn_api.football import League

In [2]:
LEAGUE_ID = 35924907
SEASON = 2026
THROUGH_WEEK = 10

ESPN_S2 = ""
SWID = ""

OUT = Path("data")
PREFIX = f"espn_{LEAGUE_ID}_"
print("Saving to", OUT.resolve())

Saving to /Users/kylesmith/Desktop/Projects/WFR/League/data


In [3]:
league = League(league_id=LEAGUE_ID, year=SEASON, espn_s2=ESPN_S2, swid=SWID)
print(league.settings.name)
print(len(league.teams), "teams · current week", league.current_week)
print("Past seasons:", league.previousSeasons)

WHS ‘23
12 teams · current week 12
Past seasons: [2020, 2021, 2022, 2023, 2024, 2025]


In [4]:
SLOT = {"RB/WR/TE": "FLEX", "BE": "BN", "D/ST": "DEF", "OP": "SUPERFLEX", "WR/TE": "WR/TE", "RB/WR": "RB/WR"}
POS = {"D/ST": "DEF"}

def manager(team):
    # Newer versions give a list of owner records; older ones give a single name
    owners = getattr(team, "owners", None) or []
    names = []
    for o in owners:
        if isinstance(o, dict):
            name = f"{o.get('firstName', '')} {o.get('lastName', '')}".strip() or o.get("displayName", "")
            names.append(name)
        else:
            names.append(str(o))
    if not names and getattr(team, "owner", None):
        names.append(str(team.owner))
    return " & ".join(n for n in names if n)

def team_id(team):
    # Bye weeks come back as 0 or None instead of a team
    return getattr(team, "team_id", None)

def seasons_to_pull():
    return sorted(set(league.previousSeasons) | {SEASON})

def load(year):
    if year == SEASON:
        return league
    try:
        return League(league_id=LEAGUE_ID, year=year, espn_s2=ESPN_S2, swid=SWID)
    except Exception as e:
        print(f"{year}: skipped, ESPN wouldn't share it ({type(e).__name__})")
        return None

In [5]:
s = league.settings
rows = [
    ("name", s.name),
    ("teams", s.team_count),
    ("regular_season_weeks", s.reg_season_count),
    ("playoff_teams", s.playoff_team_count),
    ("playoff_matchup_length", getattr(s, "playoff_matchup_period_length", "")),
    ("scoring_type", getattr(s, "scoring_type", "")),
    ("tie_rule", s.tie_rule),
    ("playoff_seed_tie_rule", s.playoff_seed_tie_rule),
    ("keepers", s.keeper_count),
    ("faab", s.faab),
]
rows += [(f"slot_{SLOT.get(k, k)}", v) for k, v in s.position_slot_counts.items() if v]
rows += [(f"score_{f['abbr']}", f["points"]) for f in s.scoring_format]
settings = pd.DataFrame(rows, columns=["setting", "value"])
settings.to_csv(OUT / f"{PREFIX}settings.csv", index=False)
settings.head(20)

,setting,value
0,name,WHS ‘23
1,teams,12
2,regular_season_weeks,14
3,playoff_teams,8
4,playoff_matchup_length,1
5,scoring_type,H2H_POINTS
6,tie_rule,SLOT_POINTS
7,playoff_seed_tie_rule,TOTAL_POINTS_SCORED
8,keepers,0
9,faab,False


In [6]:
teams_rows, sched_rows, lineup_rows, draft_rows, move_rows = [], [], [], [], []

for year in seasons_to_pull():
    lg = load(year)
    if lg is None:
        continue
    last_week = THROUGH_WEEK if year == SEASON else lg.finalScoringPeriod
    print(f"{year}: {len(lg.teams)} teams, weeks 1 to {last_week}")

    # Teams, with each season's final place
    for t in lg.teams:
        teams_rows.append(dict(season=year, team_id=t.team_id, team_name=t.team_name, abbrev=t.team_abbrev,
                               manager=manager(t), wins=t.wins, losses=t.losses, ties=t.ties,
                               points_for=t.points_for, points_against=t.points_against,
                               seed=t.standing, final_place=t.final_standing if year != SEASON else None))

    # Scores and lineups, week by week
    cache = {}
    for week in range(1, last_week + 1):
        has_lineups = year >= 2019               # ESPN has no lineups before 2019
        games = lg.box_scores(week, player_team_cache=cache) if has_lineups else lg.scoreboard(week)
        for g in games:
            sched_rows.append(dict(season=year, week=week,
                                   home_team_id=team_id(g.home_team), away_team_id=team_id(g.away_team),
                                   home_score=g.home_score, away_score=g.away_score,
                                   bracket=getattr(g, "matchup_type", "NONE")))
            if not has_lineups:
                continue
            for side, lineup in (("home", g.home_lineup), ("away", g.away_lineup)):
                tid = team_id(getattr(g, f"{side}_team"))
                if tid is None:
                    continue
                for p in lineup:
                    lineup_rows.append(dict(season=year, week=week, team_id=tid, player_id=p.playerId,
                                            player_name=p.name, nfl_team=p.proTeam,
                                            position=POS.get(p.position, p.position),
                                            slot=SLOT.get(p.slot_position, p.slot_position),
                                            points=p.points, projected=p.projected_points,
                                            injury=getattr(p, "injuryStatus", "")))
        time.sleep(0.3)                          # be polite to ESPN

    # Draft, with positions looked up in batches
    picks = lg.draft or []
    ids = [p.playerId for p in picks]
    pos_of = {}
    for i in range(0, len(ids), 50):
        found = lg.player_info(playerId=ids[i:i + 50]) or []
        found = found if isinstance(found, list) else [found]
        pos_of.update({pl.playerId: POS.get(pl.position, pl.position) for pl in found})
    per_round = lg.settings.team_count
    for p in picks:
        draft_rows.append(dict(season=year, round=p.round_num, round_pick=p.round_pick,
                               overall=(p.round_num - 1) * per_round + p.round_pick,
                               team_id=team_id(p.team), player_id=p.playerId, player_name=p.playerName,
                               position=pos_of.get(p.playerId, ""), keeper=bool(p.keeper_status)))

    # Adds, drops, and trades; one ID per transaction so both sides of a trade stay together
    if year >= 2019:
        offset, n = 0, 0
        while True:
            batch = lg.recent_activity(size=25, offset=offset)
            if not batch:
                break
            for a in batch:
                n += 1
                for team, action, player, bid in a.actions:
                    move_rows.append(dict(season=year, move_id=f"{year}-{n}",
                                          date=pd.to_datetime(a.date, unit="ms"),
                                          team_id=team_id(team), action=action,
                                          player_id=getattr(player, "playerId", player),
                                          player_name=getattr(player, "name", ""),
                                          position=POS.get(getattr(player, "position", ""), getattr(player, "position", "")),
                                          bid=bid))
            offset += 25
            time.sleep(0.3)

teams_df = pd.DataFrame(teams_rows).astype({"final_place": "Int64"})
sched_df = pd.DataFrame(sched_rows).astype({"home_team_id": "Int64", "away_team_id": "Int64"})
lineups_df = pd.DataFrame(lineup_rows)
draft_df = pd.DataFrame(draft_rows)
moves_df = pd.DataFrame(move_rows)
print("done")

2020: 12 teams, weeks 1 to 16
2021: 12 teams, weeks 1 to 17
2022: 12 teams, weeks 1 to 17
2023: 12 teams, weeks 1 to 17
2024: 12 teams, weeks 1 to 17
2025: 12 teams, weeks 1 to 17
2026: 12 teams, weeks 1 to 10
done


In [7]:
files = {"teams": teams_df, "schedule": sched_df, "lineups": lineups_df, "draft": draft_df, "transactions": moves_df}
for name, df in files.items():
    df.to_csv(OUT / f"{PREFIX}{name}.csv", index=False)

check = pd.DataFrame({name: df.groupby("season").size() if len(df) else pd.Series(dtype=int)
                      for name, df in files.items()}).fillna(0).astype(int)
print("Rows per season")
display(check)

# Every finished week should have one game per pair of teams
games_per_week = sched_df.groupby(["season", "week"]).size()
odd = games_per_week[games_per_week < league.settings.team_count // 2]
if len(odd):
    print("Weeks with fewer games than expected (playoff weeks are normal here):")
    display(odd.reset_index(name="games"))

Rows per season


,teams,schedule,lineups,draft,transactions
season,,,,,
2020,12,96,3170,192,0
2021,12,102,3461,192,0
2022,12,102,3551,192,0
2023,12,102,3424,192,0
2024,12,102,3443,192,0
2025,12,102,3512,192,0
2026,12,60,2109,192,572
